### Build a Question Answering application over a Graph Database

In [5]:
NEO4J_URI="neo4j+s://30872071.databases.neo4j.io"
NEO4J_USERNAME="30872071"
NEO4J_PASSWORD="wtyLQHg2I5GuMkQPF8inwMjPQMCGQ7JdyJ92dRHwXMQ"
NEO4J_DATABASE="30872071"


In [6]:
import os
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD

In [7]:
print("URI:", NEO4J_URI)
print("USERNAME:", NEO4J_USERNAME)
print("DATABASE:", NEO4J_DATABASE)

URI: neo4j+s://30872071.databases.neo4j.io
USERNAME: 30872071
DATABASE: 30872071


In [8]:
from langchain_community.graphs import Neo4jGraph
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)
print("Graph connection established successfully.")

Unable to retrieve routing information


ValueError: Could not connect to Neo4j database. Please ensure that the url is correct

In [31]:
## Dataset Moview 
moview_query="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') | 
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') | 
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') | 
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))


"""

In [32]:
moview_query

"\nLOAD CSV WITH HEADERS FROM\n'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row\n\nMERGE(m:Movie{id:row.movieId})\nSET m.released = date(row.released),\n    m.title = row.title,\n    m.imdbRating = toFloat(row.imdbRating)\nFOREACH (director in split(row.director, '|') | \n    MERGE (p:Person {name:trim(director)})\n    MERGE (p)-[:DIRECTED]->(m))\nFOREACH (actor in split(row.actors, '|') | \n    MERGE (p:Person {name:trim(actor)})\n    MERGE (p)-[:ACTED_IN]->(m))\nFOREACH (genre in split(row.genres, '|') | \n    MERGE (g:Genre {name:trim(genre)})\n    MERGE (m)-[:IN_GENRE]->(g))\n\n\n"

In [39]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

In [40]:
from langchain_groq import ChatGroq
llm=ChatGroq(groq_api_key=groq_api_key,model_name="Gemma2-9b-It")
llm

ChatGroq(profile={}, client=<groq.resources.chat.completions.Completions object at 0x0000029D7E68ED20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000029D7E6DBBF0>, model_name='Gemma2-9b-It', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [44]:
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(graph=driver, llm=llm, verbose=True, allow_dangerous_requests=True)
chain

AttributeError: 'Neo4jDriver' object has no attribute 'get_structured_schema'

In [ ]:
response=chain.invoke({"query":"Who was the director of the movie Casino"})
response

In [ ]:
response=chain.invoke({"query":"Who were the actors of the movie Casino"})
response

In [ ]:
response=chain.invoke({"query":"How many artists are there?"})
response

In [ ]:
response=chain.invoke({"query":"How many movies has Tom Hanks acted in"})
response